In [1]:
import os
import pandas as pd
from gensim.models import KeyedVectors
from scipy.stats import spearmanr

# --- ファイルパスの設定 ---
MODEL_FILE = "GoogleNews-vectors-negative300.bin.gz"
EVAL_FILE = "combined.csv"

# --- ファイルの存在チェック ---
if not os.path.exists(MODEL_FILE):
    print(f"エラー: モデルファイルが見つかりません: {MODEL_FILE}")
elif not os.path.exists(EVAL_FILE):
    print(f"エラー: 評価ファイルが見つかりません: {EVAL_FILE}")
else:
    try:
        # --- モデルの読み込み ---
        print("単語ベクトルモデルを読み込んでいます... (時間がかかる場合があります)")
        model = KeyedVectors.load_word2vec_format(MODEL_FILE, binary=True)
        print("モデルの読み込みが完了しました。")

        # --- 評価データの読み込み ---
        print("評価データ(combined.csv)を読み込んでいます...")
        df = pd.read_csv(EVAL_FILE)
        print("評価データの読み込みが完了しました。")

        human_scores = []
        model_scores = []

        # --- 各単語ペアの類似度を計算 ---
        print("類似度を計算し、相関係数を算出します...")
        for index, row in df.iterrows():
            word1 = row['Word 1']
            word2 = row['Word 2']
            human_score = row['Human (mean)']

            # モデルの語彙に単語が存在するかチェック
            if word1 in model and word2 in model:
                # コサイン類似度を計算
                model_similarity = model.similarity(word1, word2)
                human_scores.append(human_score)
                model_scores.append(model_similarity)
            else:
                # 語彙にない単語ペアはスキップ
                # print(f"Skipping pair with OOV word: {word1}, {word2}")
                pass

        # --- スピアマン相関係数の計算 ---
        if len(human_scores) > 1:
            correlation, p_value = spearmanr(human_scores, model_scores)
            print("\n--- 計算結果 ---")
            print(f"評価した単語ペアの数: {len(human_scores)}")
            print(f"スピアマン相関係数: {correlation:.4f}")
        else:
            print("相関係数を計算できるほどの有効なデータがありませんでした。")

    except Exception as e:
        print(f"処理中に予期せぬエラーが発生しました: {e}")

単語ベクトルモデルを読み込んでいます... (時間がかかる場合があります)
モデルの読み込みが完了しました。
評価データ(combined.csv)を読み込んでいます...
評価データの読み込みが完了しました。
類似度を計算し、相関係数を算出します...

--- 計算結果 ---
評価した単語ペアの数: 353
スピアマン相関係数: 0.7000
